# TE family x genomic-location tables + TE -> location mapping, for the 10 TE
# families, using Ensembl v113 overlap-based labels (genic-first priority)

In [1]:
1

[1] 1

In [1]:
library(tidyverse)
library(synExtra)
library(qs)
library(data.table)
library(rtracklayer)
library(GenomicRanges)
library(GenomeInfoDb)

── Attaching core tidyverse packages ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
qs 0.27.3. Announcement: https://github.com/qsbase/qs/issues/103


Attaching package: ‘data.table’


The following objects are masked from ‘packag

In [15]:
synapser::synLogin()
DATA_DIR <- path.expand("~/projects/tdp-43/scripts/analysis/final_dsRNA/liu_data/data")
dir.create(DATA_DIR, showWarnings = FALSE, recursive = TRUE)
syn <- synDownloader(DATA_DIR, .cache = TRUE)

NULL

In [16]:
# ---- parameters ----
REPEATS_QS_SYN <- "syn64829593"   # liu_repeat_counts_raw.qs
RM_RAW_SYN     <- "syn55256576"           # RepeatMasker raw (class_family, rep_name)
META_SYN       <- "syn36639762"           # run-level metadata
GTF_SYN        <- NA_character_           # Ensembl 113 GTF synID
GTF_URL        <- "https://ftp.ensembl.org/pub/release-113/gtf/homo_sapiens/Homo_sapiens.GRCh38.113.gtf.gz"
UPLOAD_PARENT  <- "syn36639270"           # outputs need to go in the "TE_region_counts" subfolder
SCRIPT_REF     <- NA_character_           # ??


In [11]:
stopifnot("Set REPEATS_QS_SYN to the real synID" = grepl("^syn[0-9]+$", REPEATS_QS_SYN))


In [12]:
fams10 <- c("Retroposon/SVA","SINE/Alu","SINE/MIR","LINE/L1","LINE/L2",
            "LTR","LTR/ERV1","LTR/ERVK","LTR/ERVL","LTR/ERVL-MaLR")
region_levels <- c("Promoter (<=3kb)","5' UTR","3' UTR","Exon","Intron","Downstream","Distal Intergenic")


In [7]:
e <- synapser::synGet(REPEATS_QS_SYN, downloadFile = FALSE)
e$properties$name
e$properties$concreteType

[1] "RNA-seq"

[1] "org.sagebionetworks.repo.model.Folder"

In [8]:
REPEATS_QS_SYN <- synPluck("syn36639270", "Repeats", "liu_repeat_counts_raw.qs")

In [9]:
synChildren(synPluck("syn36639270", "Repeats"))

de_te_location_counts.xlsx           df_excess_TE_in_TDP-43.csv 
                       "syn74417245"                        "syn68394805" 
            liu_repeat_counts_raw.qs            liu_repeat_counts_raw.qs2 
                       "syn64829593"                        "syn74844895" 
repeat_tdp43_pos_vs_neg_deseq.csv.gz 
                       "syn68394812"

In [17]:
# featureCounts object 
fc_raw <- syn(REPEATS_QS_SYN) %>% qread()
stopifnot(identical(rownames(fc_raw$counts), as.character(fc_raw$annotation$GeneID)))
counts_mat <- as.data.table(fc_raw$counts, keep.rownames = "GeneID")      #  in here GeneID + one col per BAM
ann <- as.data.table(fc_raw$annotation)                                   ## GeneID, Chr, Start, End, Strand, Length
ann[, `:=`(Chr = as.character(Chr), Start = as.integer(Start),
           End = as.integer(End), Strand = as.character(Strand))]
stopifnot(!anyDuplicated(ann$GeneID), !anyNA(ann$Start), !anyNA(ann$End))  ### NA => multi-feature meta-features


In [18]:
# RepeatMasker -> class_family + rep_name 
rm_raw <- syn(RM_RAW_SYN) %>% read_csv(show_col_types = FALSE) %>% as.data.table()
stopifnot(
  "rm_raw / featureCounts row counts differ" = nrow(rm_raw) == nrow(ann),
  "chromosomes not row-aligned"              = all(ann$Chr == rm_raw$chromosome),
  "end coordinates not row-aligned"          = all(ann$End == rm_raw$end),
  "start coordinates not row-aligned"        = all((ann$Start - rm_raw$start) %in% c(0, 1)),  
  # in here droppingg the next line if the GeneID pattern in the featureCounts script is different
  "GeneID != rep_name_<row>"                 = all(ann$GeneID == paste0(rm_raw$rep_name, "_", seq_len(nrow(rm_raw))))
)
ann[, `:=`(class_family = rm_raw$class_family, rep_name = rm_raw$rep_name)]
stopifnot("Some of fams10 are absent from class_family (naming mismatch?)" = all(fams10 %in% ann$class_family))


In [20]:
setdiff(c("Run","patient_sample_id","tdp_43_status","size_factor"), names(liu_meta))
names(liu_meta)
head(liu_meta, 3)

[1] "patient_sample_id" "tdp_43_status"     "size_factor"

[1] "Run"                 "Assay Type"          "AvgSpotLen"         
 [4] "Bases"               "BioProject"          "BioSample"          
 [7] "Bytes"               "Cell_type"           "Center Name"        
[10] "Consent"             "DATASTORE filetype"  "DATASTORE provider" 
[13] "DATASTORE region"    "Diagnosis"           "Experiment"         
[16] "gender"              "GEO_Accession (exp)" "Instrument"         
[19] "LibraryLayout"       "LibrarySelection"    "LibrarySource"      
[22] "Organism"            "Platform"            "ReleaseDate"        
[25] "Sample Name"         "source_name"         "SRA Study"          
[28] "Tissue"              "genotype/variation"

Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,Bytes,Cell_type,Center Name,Consent,⋯,LibrarySelection,LibrarySource,Organism,Platform,ReleaseDate,Sample Name,source_name,SRA Study,Tissue,genotype/variation
<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<dttm>,<chr>,<chr>,<chr>,<chr>,<chr>
SRR8571937,RNA-Seq,240,34996399600,PRJNA522295,SAMN10926441,14893314011,NeuN positive,GEO,public,⋯,cDNA,TRANSCRIPTOMIC,Homo sapiens,ILLUMINA,2019-03-16,GSM3603263,Nuclei,SRP185789,Frontal cortex,TDP-43 positive
SRR8571938,RNA-Seq,239,36288552250,PRJNA522295,SAMN10926440,15501584879,NeuN positive,GEO,public,⋯,cDNA,TRANSCRIPTOMIC,Homo sapiens,ILLUMINA,2019-03-16,GSM3603264,Nuclei,SRP185789,Frontal cortex,TDP-43 negative
SRR8571939,RNA-Seq,239,37159091950,PRJNA522295,SAMN10926438,15906400112,NeuN positive,GEO,public,⋯,cDNA,TRANSCRIPTOMIC,Homo sapiens,ILLUMINA,2019-03-16,GSM3603265,Nuclei,SRP185789,Frontal cortex,TDP-43 positive


In [21]:
nrow(liu_meta)
liu_meta[, .(Run, `Sample Name`, BioSample, Cell_type, Diagnosis, gender, source_name, `genotype/variation`)]
colnames(fc_raw$counts)

[1] 16

Run,Sample Name,BioSample,Cell_type,Diagnosis,gender,source_name,genotype/variation
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
SRR8571937,GSM3603263,SAMN10926441,NeuN positive,FTD,female,Nuclei,TDP-43 positive
SRR8571938,GSM3603264,SAMN10926440,NeuN positive,FTD,female,Nuclei,TDP-43 negative
SRR8571939,GSM3603265,SAMN10926438,NeuN positive,FTD-ALS,male,Nuclei,TDP-43 positive
SRR8571940,GSM3603266,SAMN10926436,NeuN positive,FTD-ALS,male,Nuclei,TDP-43 negative
SRR8571941,GSM3603267,SAMN10926434,NeuN positive,FTD,female,Nuclei,TDP-43 positive
SRR8571942,GSM3603268,SAMN10926433,NeuN positive,FTD,female,Nuclei,TDP-43 negative
SRR8571943,GSM3603269,SAMN10926431,unsorted,FTD,female,Nuclei,NA
SRR8571944,GSM3603270,SAMN10926429,NeuN positive,FTD,female,Nuclei,TDP-43 positive
SRR8571945,GSM3603271,SAMN10926428,NeuN positive,FTD,female,Nuclei,TDP-43 negative


[1] "SRR8571937.sorted.bam" "SRR8571938.sorted.bam" "SRR8571939.sorted.bam"
 [4] "SRR8571940.sorted.bam" "SRR8571941.sorted.bam" "SRR8571942.sorted.bam"
 [7] "SRR8571943.sorted.bam" "SRR8571944.sorted.bam" "SRR8571945.sorted.bam"
[10] "SRR8571946.sorted.bam" "SRR8571947.sorted.bam" "SRR8571948.sorted.bam"
[13] "SRR8571949.sorted.bam" "SRR8571950.sorted.bam" "SRR8571951.sorted.bam"
[16] "SRR8571952.sorted.bam"

In [22]:
# run-level metadata
SIZE_FACTOR_SYN <- NA_character_  

In [23]:
liu_meta <- syn(META_SYN) %>% read_csv(show_col_types = FALSE) %>% as.data.table()
need <- c("Run", "Sample Name", "Cell_type", "Diagnosis", "gender", "genotype/variation")
if (length(miss <- setdiff(need, names(liu_meta))))
  stop("Metadata is missing column(s): ", paste(miss, collapse = ", "))
stopifnot(!anyDuplicated(liu_meta$Run))

Using cached GSE126542.csv



In [24]:
# keeping the 14 sorted NeuN+ libraries; the 2 unsorted libraries have no TDP-43 status - need to drop them
smap <- liu_meta[Cell_type == "NeuN positive" & !is.na(`genotype/variation`),
                 .(Run, GSM = `Sample Name`, Diagnosis, gender, status = `genotype/variation`)]
stopifnot(nrow(smap) == 14,
          all(smap$status %in% c("TDP-43 positive", "TDP-43 negative")))
smap[, Group := fifelse(status == "TDP-43 positive", "TDP43+", "TDP43-")]

In [25]:
# subject: each subject is deposited as consecutive positive/negative GSMs (S1 tillk the S7)
setorder(smap, GSM)
smap[, Sample := rep(seq_len(.N / 2), each = 2)]
chk <- smap[, .(n = .N, n_grp = uniqueN(Group), n_dx = uniqueN(Diagnosis), n_sex = uniqueN(gender)), by = Sample]
stopifnot("pairing check failed: each subject needs one TDP43+ and one TDP43- library with matching diagnosis/sex" =
            all(chk$n == 2 & chk$n_grp == 2 & chk$n_dx == 1 & chk$n_sex == 1))

In [26]:
samples <- smap$Run

In [27]:
# size factors calc
if (!is.na(SIZE_FACTOR_SYN)) {
  sf   <- syn(SIZE_FACTOR_SYN) %>% read_csv(show_col_types = FALSE) %>% as.data.table()
  smap <- merge(smap, sf[, .(Run, size_factor)], by = "Run", sort = FALSE)
  stopifnot(nrow(smap) == 14)
} else {
  # doing here the DESeq2 median-of-ratios on the repeat counts, using loci detected in all 14 libraries we have
  m  <- as.matrix(fc_raw$counts[, paste0(samples, ".sorted.bam")])
  m  <- m[rowSums(m > 0) == ncol(m), , drop = FALSE]
  lg <- log(m)
  smap[, size_factor := exp(apply(lg - rowMeans(lg), 2, median))]
  rm(m, lg); invisible(gc())
}
stopifnot("invalid size factors" = all(is.finite(smap$size_factor) & smap$size_factor > 0))

In [28]:
samples <- smap$Run
smap[, .(Run, Sample, Group, Diagnosis, gender, size_factor)]   

Run,Sample,Group,Diagnosis,gender,size_factor
<chr>,<int>,<chr>,<chr>,<chr>,<dbl>
SRR8571937,1,TDP43+,FTD,female,0.9396230
SRR8571938,1,TDP43-,FTD,female,0.9964168
SRR8571939,2,TDP43+,FTD-ALS,male,1.3010267
SRR8571940,2,TDP43-,FTD-ALS,male,1.1539487
SRR8571941,3,TDP43+,FTD,female,1.0280307
SRR8571942,3,TDP43-,FTD,female,1.0010587
SRR8571944,4,TDP43+,FTD,female,0.9670999
SRR8571945,4,TDP43-,FTD,female,0.9943820
SRR8571947,5,TDP43+,FTD,male,1.0410408


In [29]:

gtf_path <- if (!is.na(GTF_SYN)) syn(GTF_SYN) else {
  dest <- "Homo_sapiens.GRCh38.113.gtf.gz"
  if (!file.exists(dest)) {
    options(timeout = max(3600, getOption("timeout")))
    tmp <- paste0(dest, ".part")
    download.file(GTF_URL, tmp, mode = "wb")
    file.rename(tmp, dest)
  }
  dest
}

In [30]:
gtf <- import(gtf_path)

# Harmonizing seqnames to UCSC style (1 -> chr1, MT -> chrM)
std      <- c(as.character(1:22), "X", "Y", "MT")
gtf      <- keepSeqlevels(gtf, intersect(std, seqlevels(gtf)), pruning.mode = "coarse")
newnames <- paste0("chr", seqlevels(gtf)); newnames[newnames == "chrMT"] <- "chrM"
gtf      <- renameSeqlevels(gtf, setNames(newnames, seqlevels(gtf)))
std_chr  <- seqlevels(gtf)


## Ensembl leaves gene_name NA for unnamed genes -> falling back to gene_id
gtf$label <- ifelse(is.na(gtf$gene_name), gtf$gene_id, gtf$gene_name)


In [31]:
tx      <- gtf[gtf$type == "transcript"]
exons   <- reduce(gtf[gtf$type == "exon"], ignore.strand = TRUE)

In [32]:
cds_gr  <- gtf[gtf$type == "CDS"]
u5      <- reduce(gtf[gtf$type == "five_prime_utr"],  ignore.strand = TRUE)
u3      <- reduce(gtf[gtf$type == "three_prime_utr"], ignore.strand = TRUE)
genes   <- gtf[gtf$type == "gene"]
genes_r <- reduce(genes, ignore.strand = TRUE)
prom    <- reduce(suppressWarnings(promoters(tx, upstream = 3000, downstream = 3000)), ignore.strand = TRUE)
downs   <- reduce(suppressWarnings(flank(tx, width = 300, start = FALSE)), ignore.strand = TRUE)
introns <- GenomicRanges::setdiff(genes_r, exons)
tss     <- resize(tx, 1, fix = "start")

In [33]:
rm(gtf); invisible(gc())

In [34]:
te <- GRanges(ann$Chr, IRanges(ann$Start, ann$End),
              strand = ifelse(ann$Strand %in% c("+","-"), ann$Strand, "*"))

In [35]:
ov <- function(x, y) suppressWarnings(overlapsAny(x, y, ignore.strand = TRUE))
o_cds <- ov(te, cds_gr); o_u5 <- ov(te, u5); o_u3 <- ov(te, u3); o_ex <- ov(te, exons)
o_in  <- ov(te, introns); o_pr <- ov(te, prom); o_dn <- ov(te, downs)
ann[, region := fifelse(o_cds, "Exon",
             fifelse(o_u5, "5' UTR",
             fifelse(o_u3, "3' UTR",
             fifelse(o_ex, "Exon",
             fifelse(o_in, "Intron",
             fifelse(o_pr, "Promoter (<=3kb)",
             fifelse(o_dn, "Downstream", "Distal Intergenic")))))))]

In [36]:
## TEs on scaffolds/ALT contigs here cannot be annotated (GTF pruned to main chromosomes only for now ):
# leaving region as NA instead of calling them "Distal Intergenic" in here the code
ann[!(Chr %in% std_chr), region := NA_character_]

In [37]:
# mapping extras: overlap fractions, orientation, nearest gene + also distance to TSS
frac <- function(y) {
  h  <- suppressWarnings(findOverlaps(te, y, ignore.strand = TRUE))
  w  <- width(pintersect(te[queryHits(h)], y[subjectHits(h)], ignore.strand = TRUE))
  bp <- numeric(length(te))
  s  <- rowsum(w, queryHits(h), reorder = FALSE)
  bp[as.integer(rownames(s))] <- s[, 1]
  bp / width(te)
}
ann[, `:=`(frac_exonic = round(frac(exons), 4), frac_intronic = round(frac(introns), 4))]


In [38]:
hg     <- suppressWarnings(findOverlaps(te, genes, ignore.strand = TRUE))
te_str <- as.character(strand(te))[queryHits(hg)]
known  <- te_str != "*"

In [39]:
same   <- te_str == as.character(strand(genes))[subjectHits(hg)]
sn <- logical(length(te)); an <- logical(length(te)); has_gene <- logical(length(te))

In [40]:
sn[unique(queryHits(hg)[known & same])]  <- TRUE
an[unique(queryHits(hg)[known & !same])] <- TRUE
has_gene[unique(queryHits(hg))]          <- TRUE

In [41]:
ann[, orientation := fifelse(sn & an, "both",
                     fifelse(sn, "sense",
                     fifelse(an, "antisense",
                     fifelse(has_gene, "unstranded", "no_gene"))))]

In [42]:
ng <- suppressWarnings(nearest(te, genes, ignore.strand = TRUE))         # for the nearest gene body
ann[, nearest_gene := genes$label[ng]]
dt <- suppressWarnings(distanceToNearest(te, tss, ignore.strand = TRUE)) # nearest TSS (may be another gene)
ann[, `:=`(distanceToTSS = NA_integer_, nearest_tss_gene = NA_character_)]
ann[queryHits(dt), `:=`(distanceToTSS    = mcols(dt)$distance,
                        nearest_tss_gene = tss$label[subjectHits(dt)])]

In [43]:
# map each Run to exactly one count column 
bamcol <- setdiff(names(counts_mat), "GeneID")

In [44]:
run_to_col <- vapply(samples, function(r) {
  hit <- bamcol[str_detect(bamcol, regex(paste0("(?<![A-Za-z0-9])", r, "(?![0-9])")))]
  if (length(hit) != 1)
    stop("Run ", r, " matches ", length(hit), " count columns (expected exactly 1)")
  hit
}, character(1))
stopifnot(!anyDuplicated(run_to_col))

In [45]:
if (length(unused <- setdiff(bamcol, run_to_col)))
  message(length(unused), " BAM column(s) not in the metadata were ignored: ", paste(unused, collapse = ", "))

2 BAM column(s) not in the metadata were ignored: SRR8571943.sorted.bam, SRR8571946.sorted.bam



In [47]:
cnt <- counts_mat[, c("GeneID", unname(run_to_col)), with = FALSE]
setnames(cnt, c("GeneID", samples))
cnt[, (samples) := lapply(.SD, as.numeric), .SDcols = samples]            # for avoid integer overflow in sums
cnt <- merge(cnt, ann[, .(GeneID, class_family, region)], by = "GeneID", sort = FALSE)
cnt <- cnt[class_family %in% fams10]


In [48]:
unplaced <- cnt[is.na(region)]
if (nrow(unplaced))
  message(nrow(unplaced), " loci on non-standard contigs excluded from region tables (",
          signif(100 * sum(as.matrix(unplaced[, ..samples])) / sum(as.matrix(cnt[, ..samples])), 3),
          "% of 10-family reads)")


In [49]:
agg  <- cnt[!is.na(region), lapply(.SD, sum), by = .(class_family, region), .SDcols = samples]
long <- melt(agg, id.vars = c("class_family","region"), variable.name = "Run",
             value.name = "sum_raw_counts", variable.factor = FALSE)
setnames(long, c("class_family","region"), c("TE_family","Genomic_location"))
full <- CJ(TE_family = fams10, Genomic_location = region_levels, Run = samples, unique = TRUE)
long <- merge(full, long, by = c("TE_family","Genomic_location","Run"), all.x = TRUE)
long[is.na(sum_raw_counts), sum_raw_counts := 0]
long <- merge(long, smap, by = "Run")
long[, sum_normalized_counts := sum_raw_counts / size_factor]
long[, TE_family := factor(TE_family, levels = fams10)]
long[, Genomic_location := factor(Genomic_location, levels = region_levels)]

In [50]:
bysample <- long[, .(Group, Sample, Run, TE_family, Genomic_location, size_factor,
                     sum_raw_counts = round(sum_raw_counts,2),
                     sum_normalized_counts = round(sum_normalized_counts,2))]
setorder(bysample, TE_family, Genomic_location, Sample, Group)
fwrite(bysample, "te_family_region_counts_by_sample_ensembl113.csv")


In [57]:
dim(bysample)

[1] 980   8

In [58]:
head(bysample)

Group,Sample,Run,TE_family,Genomic_location,size_factor,sum_raw_counts,sum_normalized_counts
<chr>,<int>,<chr>,<fct>,<fct>,<dbl>,<dbl>,<dbl>
TDP43+,1,SRR8571937,Retroposon/SVA,Promoter (<=3kb),0.9396230,6190,6587.75
TDP43-,1,SRR8571938,Retroposon/SVA,Promoter (<=3kb),0.9964168,6469,6492.26
TDP43+,2,SRR8571939,Retroposon/SVA,Promoter (<=3kb),1.3010267,7114,5467.99
TDP43-,2,SRR8571940,Retroposon/SVA,Promoter (<=3kb),1.1539487,6300,5459.51
TDP43+,3,SRR8571941,Retroposon/SVA,Promoter (<=3kb),1.0280307,7329,7129.16
TDP43-,3,SRR8571942,Retroposon/SVA,Promoter (<=3kb),1.0010587,8621,8611.88


In [51]:
bygroup <- long[, .(sum_raw_counts = round(sum(sum_raw_counts),2),
                    mean_raw_per_sample = round(mean(sum_raw_counts),2),
                    sum_normalized_counts = round(sum(sum_normalized_counts),2),
                    mean_normalized_per_sample = round(mean(sum_normalized_counts),2)),
                 by = .(Group, TE_family, Genomic_location)]
setorder(bygroup, TE_family, Genomic_location, Group)
fwrite(bygroup, "te_family_region_counts_by_group_ensembl113.csv")


In [59]:
dim(bygroup)

[1] 140   7

In [60]:
head(bygroup)

Group,TE_family,Genomic_location,sum_raw_counts,mean_raw_per_sample,sum_normalized_counts,mean_normalized_per_sample
<chr>,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
TDP43+,Retroposon/SVA,Promoter (<=3kb),47000,6714.29,45785.61,6540.80
TDP43-,Retroposon/SVA,Promoter (<=3kb),51931,7418.71,49847.22,7121.03
TDP43+,Retroposon/SVA,5' UTR,4193,599.00,4041.91,577.42
TDP43-,Retroposon/SVA,5' UTR,4556,650.86,4326.83,618.12
TDP43+,Retroposon/SVA,3' UTR,22637,3233.86,22050.10,3150.01
TDP43-,Retroposon/SVA,3' UTR,22987,3283.86,22138.23,3162.60


In [52]:
cnt[, expr_total_raw := round(rowSums(.SD),3), .SDcols = samples]
mapping <- merge(ann[class_family %in% fams10,
                     .(GeneID, seqnames = Chr, start = Start, end = End, strand = Strand,
                       TE_family = class_family, rep_name, Length, Genomic_location = region,
                       frac_exonic, frac_intronic, orientation, nearest_gene,
                       nearest_tss_gene, distanceToTSS)],
                 cnt[, .(GeneID, expr_total_raw)], by = "GeneID", sort = FALSE)

In [53]:
fwrite(mapping, "te_location_mapping_10families_ensembl113.csv.gz")

In [61]:
dim(mapping)

[1] 3959317      16

In [62]:
head(mapping)

GeneID,seqnames,start,end,strand,TE_family,rep_name,Length,Genomic_location,frac_exonic,frac_intronic,orientation,nearest_gene,nearest_tss_gene,distanceToTSS,expr_total_raw
<chr>,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<int>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<int>,<dbl>
L1MC_3,chr1,11505,11675,-,LINE/L1,L1MC,171,Exon,0.9766,0.0234,antisense,DDX11L16,DDX11L16,78,41
MIR3_5,chr1,15265,15355,-,SINE/MIR,MIR3,91,Exon,1.0000,0.0000,both,DDX11L16,WASH7P,159,80
L2a_9,chr1,18907,19048,+,LINE/L2,L2a,142,Exon,0.9577,0.0423,both,DDX11L16,WASH7P,90,1194
MLT1K_12,chr1,21949,22344,+,LTR/ERVL-MaLR,MLT1K,396,Intron,0.0000,1.0000,both,DDX11L16,WASH7P,2474,755
MIR_13,chr1,23120,23371,-,SINE/MIR,MIR,252,Exon,0.8135,0.1865,both,DDX11L16,WASH7P,1447,1108
L2b_14,chr1,23804,24038,+,LINE/L2,L2b,235,Exon,0.3106,0.6894,both,DDX11L16,WASH7P,780,636


In [54]:
# to store outputs under <project>/TE_region_counts
syn_out <- synMkdir(UPLOAD_PARENT, "TE_region_counts")

In [55]:
store_args <- list(
  c("te_family_region_counts_by_sample_ensembl113.csv",
    "te_family_region_counts_by_group_ensembl113.csv",
    "te_location_mapping_10families_ensembl113.csv.gz"),
  parentId     = syn_out,
  used         = c(REPEATS_QS_SYN, RM_RAW_SYN, META_SYN, if (!is.na(GTF_SYN)) GTF_SYN else GTF_URL),
  forceVersion = FALSE
)

In [ ]:
if (!is.na(SCRIPT_REF)) store_args$executed <- SCRIPT_REF
do.call(synStoreMany, store_args)